# Task 3 Pilot: 30-Step SFT GPU Pilot (`align-sft-pilot`)

In [ ]:
!nvidia-smi
import os, shutil, pathlib, torch, transformers, datasets

input_root = pathlib.Path('/kaggle/input')
src_dir = next(input_root.rglob('submission_adapter.py')).parent
work_repo = pathlib.Path('/kaggle/working/repo')
if work_repo.exists():
    shutil.rmtree(work_repo)
shutil.copytree(src_dir, work_repo)
os.chdir(work_repo)
print('COMMIT:', (work_repo / 'COMMIT.txt').read_text().strip())
print('PyTorch:', torch.__version__, 'Transformers:', transformers.__version__, 'Datasets:', datasets.__version__)


In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors
!python -m pip install -q -e . --no-deps


In [ ]:
# Run 30 optimizer steps on 2,000 train examples, validate on 200 examples
!PYTHONPATH=. python -m src.align.train_sft --config configs/sft_pilot.yaml


In [ ]:
# Sanity-check generation on 3 prompts from the saved pilot checkpoint
!PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path /kaggle/working/outputs/sft_pilot/best \
  --eval_set_path eval_sets/alpaca_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/sft_pilot/pilot_sanity_3.json \
  --generator_name qwen2.5-0.5b-sft-pilot \
  --dataset_name alpaca_eval \
  --limit 3 \
  --batch_size 3

# Copy metrics.jsonl and summary.json to top-level /kaggle/working for easy file filtering & download
import shutil
shutil.copy('/kaggle/working/outputs/sft_pilot/metrics.jsonl', '/kaggle/working/sft_pilot_metrics.jsonl')
shutil.copy('/kaggle/working/outputs/sft_pilot/summary.json', '/kaggle/working/sft_pilot_summary.json')
shutil.copy('/kaggle/working/outputs/sft_pilot/pilot_sanity_3.json', '/kaggle/working/sft_pilot_sanity_3.json')
# Remove checkpoint weights from pilot run so output download stays tiny
shutil.rmtree('/kaggle/working/outputs/sft_pilot/best', ignore_errors=True)
shutil.rmtree('/kaggle/working/outputs/sft_pilot/final', ignore_errors=True)
